In [3]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
class RNN(torch.nn.Module):
  def __init__(self, num_inputs, num_hiddens):
    super().__init__()
    self.num_inputs = num_inputs
    self.num_hiddens = num_hiddens
    self.rnn = torch.nn.RNN(num_inputs, num_hiddens)
  def forward(self, inputs, H=None):
    return self.rnn(inputs, H)

In [21]:
class RNNLM(torch.nn.Module):
  def __init__(self, vocab_size:int, rnn: RNN, lr=0.01):
    super().__init__()
    self.vocab_size = vocab_size
    self.rnn = rnn
    self.W_q = torch.nn.Parameter(torch.randn(rnn.num_hiddens, vocab_size))
    self.b_q = torch.nn.Parameter(torch.zeros(vocab_size))
    self.loss = torch.nn.CrossEntropyLoss()

  def training_step(self, batch):
    preds = self(*batch[:-1])
    targets = batch[-1]
    l = self.loss(preds.permute(0, 2, 1), batch[-1])
    return l

  def validation_step(self, batch):
    preds = self(*batch[:-1])
    targets = batch[-1]
    l = self.loss(preds.permute(0, 2, 1), batch[-1])
    return l

  def one_hot(self, x):
    return torch.nn.functional.one_hot(x.T, self.vocab_size).type(torch.float32)

  def output_layer(self, rnn_outputs):
    outputs = [H @ self.W_q + self.b_q for H in rnn_outputs]
    return torch.stack(outputs, 1)

  def forward(self, inputs, state=None):
    rnn_outputs, state = self.rnn(self.one_hot(inputs.to(self.W_q.device)), state)
    return self.output_layer(rnn_outputs)

  def predict(self, prefix, num_preds, vocab, device=None):
    state, outputs = None, [vocab[prefix[0]]]
    for i in range(len(prefix) + num_preds - 1):
        X = torch.tensor([[outputs[-1]]], device=device)
        embs = self.one_hot(X)
        rnn_outputs, state = self.rnn(embs, state)
        if i < len(prefix) - 1:  # Warm-up period
            outputs.append(vocab[prefix[i + 1]])
        else:  # Predict num_preds steps
            Y = self.output_layer(rnn_outputs)
            outputs.append(int(Y.argmax(2).reshape(1)))
    return ''.join([vocab.get_tokens(i) for i in outputs])

In [6]:
from collections import Counter

class Vocab:
  def __init__(self, tokens, reserved_words=[], min_freq=0):
    hist = Counter(tokens)
    self.token_freq = sorted(hist.items(), key=lambda x: x[1], reverse=True)
    self.tokens = list(set(reserved_words + ['<unk>'] + [token for token, freq in self.token_freq if freq >= min_freq]))
    self.reverse_map = {token: idx for idx, token in enumerate(self.tokens)}
  def __len__(self):
    return len(self.tokens)
  def __getitem__(self, token):
    if(isinstance(token, (list, tuple))):
      return [self.__getitem__(i) for i in token]
    else:
      return self.reverse_map.get(token, self.unk)
  def get_tokens(self, idxs):
    if(isinstance(idxs, (list, tuple))):
      return [self.tokens[i] for i in idxs]
    else:
      return self.tokens[idxs]
  @property
  def unk(self):
    return self.reverse_map['<unk>']

In [7]:
import requests

In [23]:
time_machine_text = str(requests.get('https://www.gutenberg.org/cache/epub/35/pg35.txt').text.encode('ascii', 'ignore'))

In [24]:
time_machine_vocab = Vocab(time_machine_text)

In [25]:
time_machine_data = [time_machine_vocab[char] for char in time_machine_text]

In [26]:
max(time_machine_data)

81

In [27]:
num_steps = 32

In [28]:
input_tensor = torch.stack([torch.tensor(time_machine_data[i: len(time_machine_data) - num_steps + i]) for i in range(num_steps)], 1)

In [29]:
target_tensor = torch.stack([torch.tensor(time_machine_data[i + 1: len(time_machine_data) - num_steps + i + 1]) for i in range(num_steps)], 1)

In [41]:
dataset = torch.utils.data.dataset.TensorDataset(input_tensor, target_tensor)

In [42]:
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [0.8, 0.2])

In [43]:
train_dataloader = torch.utils.data.dataloader.DataLoader(train_dataset, batch_size=1024, shuffle=True)

In [44]:
val_dataloader = torch.utils.data.dataloader.DataLoader(val_dataset, batch_size=1024, shuffle=True)

In [17]:
max_epochs = 100

In [45]:
def train_model(model, train_data, val_data, lr = 0.01, max_epochs=100, grad_clip = 1, device=torch.device('cpu')):
  model = model.to(device)
  optim = torch.optim.Adam(model.parameters(), lr=lr)
  for epoch in range(1, max_epochs + 1):
    total_loss = 0
    num_batches = 0
    model.train()
    for X, y in train_data:
      X, y = X.to(device), y.to(device)
      optim.zero_grad()
      l = model.training_step((X, y))
      l.backward()
      params = [p for p in model.parameters() if p.requires_grad]
      norm = torch.sqrt(sum(torch.sum(p ** 2) for p in params))
      if norm > grad_clip:
        for p in params:
          p.grad[:] *= grad_clip / norm
      optim.step()
      total_loss += l.item()
      num_batches += 1
    val_loss = 0
    val_num_batches = 0
    model.eval()
    for X, y in val_data:
      X, y = X.to(device), y.to(device)
      l = model.validation_step((X, y))
      val_loss += l.item()
      val_num_batches += 1
    print(f"Epoch {epoch} Train Loss: {total_loss / num_batches}")
    print(f"Epoch {epoch} Val Loss: {val_loss / val_num_batches}")
  return model

In [46]:
rnn = RNN(num_inputs=len(time_machine_vocab), num_hiddens=32)
model = RNNLM(rnn=rnn, vocab_size=len(time_machine_vocab))
model = model.to(device)
model.predict('it has', 20, time_machine_vocab, device)

'it hasENB-8888888888888888'

In [47]:
lr = 0.05

In [48]:
model = train_model(model, train_dataloader, val_dataloader, lr=lr, max_epochs=max_epochs, grad_clip=1, device=device)

Epoch 1 Train Loss: 2.4160895991179108
Epoch 1 Val Loss: 2.145714457442121
Epoch 2 Train Loss: 2.0843767227570704
Epoch 2 Val Loss: 2.0286627804360737
Epoch 3 Train Loss: 1.992517476432894
Epoch 3 Val Loss: 1.9599791270930593
Epoch 4 Train Loss: 1.9404418190564114
Epoch 4 Val Loss: 1.9245999295537064
Epoch 5 Train Loss: 1.9097126539499483
Epoch 5 Val Loss: 1.8994737223881046
Epoch 6 Train Loss: 1.886263086751926
Epoch 6 Val Loss: 1.8777843946363868
Epoch 7 Train Loss: 1.8668131455321986
Epoch 7 Val Loss: 1.862071040200024
Epoch 8 Train Loss: 1.8526489003304323
Epoch 8 Val Loss: 1.8448179378742124
Epoch 9 Train Loss: 1.8424281035464234
Epoch 9 Val Loss: 1.8459884945939227
Epoch 10 Train Loss: 1.83331952212047
Epoch 10 Val Loss: 1.8332479000091553
Epoch 11 Train Loss: 1.8270838560502223
Epoch 11 Val Loss: 1.821440653103154
Epoch 12 Train Loss: 1.8223794442744343
Epoch 12 Val Loss: 1.8177435078272006
Epoch 13 Train Loss: 1.8163517157724298
Epoch 13 Val Loss: 1.8215602171130296
Epoch 14 Tr

In [49]:
model.predict('it has', 20, time_machine_vocab, device)

'it has the story the story'